In [1]:
# Installing the 'Toolbox' (Libraries)
!pip install -q --upgrade langchain langchain-community langchain-core chromadb pypdf langchain-google-genai

### 2. Load the PDF and Split it into Chunks

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the document (assuming 'sickness and leavepolicy.pdf' is uploaded to the Colab environment)
pdf_path="/content/smart_hr_document_assistant/Sickness-And-Absence-Policy.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from the PDF.")

Created 19 chunks from the PDF.


# 3. Create the Embeddings (Turning text into numbers using Gemini)

In [6]:
import google.generativeai as genai
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)

print("Models supporting 'embedContent':")
for m in genai.list_models():
  if 'embedContent' in m.supported_generation_methods:
    print(m.name)

Models supporting 'embedContent':
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

# 4. Create/Update the Vector DB (using the chunks from your previous step)

In [9]:
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings)

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=api_key
)

# Create retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# Prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.

Context:
{context}

Question:
{question}
""")

# RAG pipeline
rag_chain = (
    {"context": retriever, "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)

# Run query
query = "Should i submit any documents if i take 3 sick leaves in a row?"
response = rag_chain.invoke(query)

print(response)

Based on the provided context, the specific number of calendar days that triggers the requirement for a self-certification form or a medical certificate is not stated. The document mentions:

*   "3.2 For sickness absences of up to [insert amount] calendar days, the self-certification form should be completed by the employee upon their return to work..."
*   "3.3 For sickness absence of more than [insert amount] calendar days, the employee must also provide a medical certificate..."

Since the "[insert amount]" is not specified, it's not possible to determine from this context whether 3 sick leaves in a row require you to submit any documents.
